# Tableau d'observabilité

Analyse les traces locales : latence, cache, erreurs, étapes coûteuses et requêtes lentes.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import load_config
cfg = load_config(ROOT / "config.yaml")
print(f"Projet : {ROOT}")


Projet : <PROJECT_ROOT>


In [2]:
import pandas as pd
from observability import load_traces, summarize_traces

events = load_traces(cfg.observability.traces_full_path)
summary = summarize_traces(events)
display(pd.Series(summary, name="valeur").to_frame())
print(f"Fichier : {cfg.observability.traces_full_path}")


,valeur
events,20
success_rate,1.0
cache_hit_rate,0.0
mean_total_seconds,0.394339
p95_total_seconds,0.422775
sources,{'evaluation.retrieval': 20}


Fichier : <PROJECT_ROOT>\data\observability\traces.jsonl


In [3]:
if events:
    flat = pd.json_normalize(events, sep=".")
    display(flat.tail(10)[[column for column in [
        "timestamp_utc", "source", "status", "question", "total_seconds",
        "retrieval_stats.total_seconds", "generation_stats.total_seconds",
        "generation_stats.cache_hit", "citation_count", "error",
    ] if column in flat.columns]])
else:
    flat = pd.DataFrame()
    print("Aucune trace pour l'instant : exécutez le benchmark ou pipeline.ask().")


,timestamp_utc,source,status,question,total_seconds,retrieval_stats.total_seconds,citation_count,error
10,2026-08-03T11:40:41.915354+00:00,evaluation.retrieval,ok,Quels sont les partenaires de l'ENSEA ?,0.347057,0.347057,0,None
11,2026-08-03T11:40:42.330488+00:00,evaluation.retrieval,ok,Où se situe l'ENSEA d'Abidjan et comment la co...,0.412940,0.412940,0,None
12,2026-08-03T11:40:42.703053+00:00,evaluation.retrieval,ok,Qu'est-ce que le Master en Data Science propos...,0.371306,0.371306,0,None
13,2026-08-03T11:40:43.127755+00:00,evaluation.retrieval,ok,Qu'est-ce que le Master en Actuariat proposé p...,0.422775,0.422775,0,None
14,2026-08-03T11:40:43.488811+00:00,evaluation.retrieval,ok,Qu'est-ce que le Master en Statistique Agricol...,0.360141,0.360141,0,None
15,2026-08-03T11:40:43.856209+00:00,evaluation.retrieval,ok,Comment fonctionne la Direction des études de ...,0.365308,0.365308,0,None
16,2026-08-03T11:40:44.281442+00:00,evaluation.retrieval,ok,Qui sont les enseignants-chercheurs de la form...,0.423424,0.423424,0,None
17,2026-08-03T11:40:44.690547+00:00,evaluation.retrieval,ok,Comment s'inscrire en tant que nouveau bacheli...,0.407490,0.407490,0,None
18,2026-08-03T11:40:45.113655+00:00,evaluation.retrieval,ok,Quel est le rôle du Bureau des admissions de l...,0.421278,0.421278,0,None
19,2026-08-03T11:40:45.530005+00:00,evaluation.retrieval,ok,Quelle est la politique de confidentialité du ...,0.415183,0.415183,0,None


In [4]:
if not flat.empty:
    timing_columns = [column for column in flat if column.startswith("retrieval_stats.") and column.endswith("seconds")]
    display(flat[timing_columns].apply(pd.to_numeric, errors="coerce").describe().T.sort_values("mean", ascending=False))


,count,mean,std,min,25%,50%,75%,max
retrieval_stats.total_seconds,20.0,0.394339,0.023985,0.347057,0.375052,0.400504,0.414202,0.423424
retrieval_stats.rerank_seconds,20.0,0.339111,0.025291,0.283376,0.323627,0.345090,0.360832,0.370708
retrieval_stats.query_embedding_seconds,20.0,0.033058,0.005130,0.027316,0.029199,0.029984,0.037116,0.044348
retrieval_stats.dense_search_seconds,20.0,0.015302,0.000657,0.014055,0.014942,0.015211,0.015822,0.016385
retrieval_stats.bm25_search_seconds,20.0,0.004203,0.000979,0.001706,0.003837,0.004229,0.004882,0.005752
retrieval_stats.vector_lookup_mmr_seconds,20.0,0.002291,0.000370,0.001660,0.002038,0.002305,0.002555,0.002869
retrieval_stats.fusion_metadata_seconds,20.0,0.000110,0.000024,0.000089,0.000094,0.000109,0.000112,0.000203


In [5]:
if not flat.empty and "total_seconds" in flat:
    slow = flat.sort_values("total_seconds", ascending=False).head(15)
    display(slow[[column for column in ["question", "source", "total_seconds", "status", "error"] if column in slow]])


,question,source,total_seconds,status,error
16,Qui sont les enseignants-chercheurs de la form...,evaluation.retrieval,0.423424,ok,None
13,Qu'est-ce que le Master en Actuariat proposé p...,evaluation.retrieval,0.422775,ok,None
18,Quel est le rôle du Bureau des admissions de l...,evaluation.retrieval,0.421278,ok,None
19,Quelle est la politique de confidentialité du ...,evaluation.retrieval,0.415183,ok,None
2,Qu'est-ce que la filière des Ingénieurs Statis...,evaluation.retrieval,0.414585,ok,None
5,Quels sont les frais de scolarité / coûts de f...,evaluation.retrieval,0.414075,ok,None
0,Quelles sont les filières de formation initial...,evaluation.retrieval,0.413438,ok,None
11,Où se situe l'ENSEA d'Abidjan et comment la co...,evaluation.retrieval,0.412940,ok,None
1,Qu'est-ce que la filière des Analystes Statist...,evaluation.retrieval,0.409499,ok,None
17,Comment s'inscrire en tant que nouveau bacheli...,evaluation.retrieval,0.407490,ok,None
